In [49]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from optuna.integration.mlflow import MLflowCallback
import optuna
import xgboost as xgb
from scipy.stats import randint, uniform
from xgboost import XGBRegressor, plot_importance
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv

# load .env if exists
load_dotenv()

# === Настройки ===
TRACKING_SERVER_HOST = os.getenv("TRACKING_SERVER_HOST")
TRACKING_SERVER_PORT = os.getenv("TRACKING_SERVER_PORT")
EXPERIMENT_NAME = os.getenv("EXPERIMENT_NAME")
REGISTRY_MODEL_NAME = os.getenv("REGISTRY_MODEL_NAME")

ML_FLOW_TRACKING_URI = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

mlflow.set_tracking_uri(ML_FLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


<Experiment: artifact_location='file:///C:\\Files\\AI\\Projects\\portfolio\\forecasting_orders_timeseries\\mlflow/artifacts/1', creation_time=1756925771708, experiment_id='1', last_update_time=1756925771708, lifecycle_stage='active', name='optuna_forecasting', tags={}>

In [50]:
df_features = pd.read_csv('..\\db_destination\\taxi_features.csv')
print(f"Размер данных: {df_features.shape}")
df_features.head()

Размер данных: (4248, 12)


,num_orders,hour_sin,hour_cos,lag_1,lag_2,lag_3,lag_24,lag_48,lag_72,lag_168,rolling_mean_24,rolling_std_24
0,143,0.000000,1.000000,94,127,108,100,42,86,124,60.041667,33.200942
1,78,0.258819,0.965926,143,94,127,121,75,176,85,61.833333,36.452073
2,65,0.500000,0.866025,78,143,94,24,36,32,71,60.041667,34.417487
3,68,0.707107,0.707107,65,78,143,66,49,51,66,61.750000,33.557543
4,60,0.866025,0.500000,68,65,78,73,30,34,43,61.833333,33.571037


In [51]:
X = df_features.drop('num_orders', axis=1)
y = df_features['num_orders']

# Используем train_test_split для разделения данных (shuffle=False для временных рядов!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, shuffle=False
)
print(f"Обучающая выборка: {X_train.shape}, тестовая: {X_test.shape}")


Обучающая выборка: (3823, 11), тестовая: (425, 11)


In [52]:
# Определяем списки признаков для пайплайна
numeric_features = ['lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_48', 'lag_72', 'lag_168',
                   'rolling_mean_24', 'rolling_std_24']
cyclic_features = ['hour_sin', 'hour_cos']


# Создаем пайплайн с обновленными признаками
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('scaler', StandardScaler())  # Масштабируем числовые признаки
        ]), numeric_features),
        ('cyclic', 'passthrough', cyclic_features)  # Циклические признаки не требуют обработки
    ],
    remainder='drop'  # Игнорируем все остальные колонки
)

In [53]:
N_TRIALS = 100       # число триалов Optuna 
CV_FOLDS = 5        # число фолдов для TimeSeriesSplit
SEED = 42           # сид для детерминированности


# (опционально) MLflow callback — если у тебя уже настроен MLflow в окружении,
# ты можешь раскомментировать блок с MLflowCallback и передать mlflc в callbacks ниже.
# from optuna.integration.mlflow import MLflowCallback
# mlflc = MLflowCallback(tracking_uri=MLFLOW_TRACKING_URI, metric_name='R2', create_experiment=False)

# Objective для Optuna
def objective(trial):
    # гиперпараметры XGBoost 
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0,
    }

    # Пайплайн: тот же preprocessor + регрессор
    model = Pipeline([
        ('preprocessor', preprocessor),   
        ('regressor', XGBRegressor(**params, enable_categorical=True))
    ])

    tscv = TimeSeriesSplit(n_splits=CV_FOLDS)
    r2_scores = []
    rmses = []

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # тренировка и предсказание
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        r2 = r2_score(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        trial.report(r2, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

        r2_scores.append(r2)
        rmses.append(rmse)

    mean_r2 = float(np.mean(r2_scores))
    mean_rmse = float(np.mean(rmses))

    # сохраняем дополнительные данные в trial (удобно смотреть в study.trials)
    trial.set_user_attr("std_r2", float(np.std(r2_scores)))
    trial.set_user_attr("rmse", mean_rmse)

    # возвращаем целью оптимизации — mean_r2 (Optuna настроен на maximize)
    return mean_r2

RUN_NAME = 'optuna_model'

# Настройка MLflow эксперимента
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

# Создаем родительский run с явным указанием вложенности
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id, nested=True) as parent_run:
    run_id = parent_run.info.run_id
    
    # Инициализация MLflow callback с nested=True
    mlflc = MLflowCallback(
        tracking_uri=f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}",
        metric_name='mean_R2',
        create_experiment=False,
        mlflow_kwargs={
            'experiment_id': experiment_id,
            'tags': {
                'mlflow.parentRunId': run_id,
                'optimizer': 'Optuna',
                'model_type': 'XGBoost',
                'cv_folds': str(CV_FOLDS),
                'n_trials': str(N_TRIALS)
            },
            'nested': True  # Ключевой параметр
        }
    )

    # Создаем study
    study = optuna.create_study(
        direction='maximize',
        study_name='xgboost_optuna',
        pruner=optuna.pruners.HyperbandPruner(
            min_resource=10,
            max_resource=600,
            reduction_factor=2
        )
    )

    # Запускаем оптимизацию
    print("Запуск подбора гиперпараметров с Optuna...")
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True,
        callbacks=[mlflc]
    )

    # Логируем лучшие параметры
    best_trial = study.best_trial
    best_rmse = best_trial.user_attrs["rmse"]
    mlflow.log_params(study.best_params)
    mlflow.log_metric("mean_R2", study.best_value)
    mlflow.log_metric("mean_RMSE", best_rmse)
    mlflow.set_tag("status", "completed")

    print(f"✅ Optuna завершена. Лучшие параметры: {study.best_params}")
    print(f"Best mean_R2: {study.best_value:.6f}")
    print(f"Best mean_RMSE: {float(np.mean([t.user_attrs['rmse'] for t in study.trials if t.value == study.best_value])):.6f}")

C:\Users\Andrey\AppData\Local\Temp\ipykernel_21456\2701270978.py:81: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2025-09-04 20:49:19,899] A new study created in memory with name: xgboost_optuna


Запуск подбора гиперпараметров с Optuna...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-09-04 20:49:20,357] Trial 0 finished with value: 0.5665218234062195 and parameters: {'n_estimators': 163, 'max_depth': 8, 'learning_rate': 0.025810671747023, 'subsample': 0.6498490505862904, 'colsample_bytree': 0.6317714598398925, 'gamma': 0.9759258789983629, 'min_child_weight': 3, 'reg_alpha': 7.014780202321949, 'reg_lambda': 8.668346024690187}. Best is trial 0 with value: 0.5665218234062195.
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/b1f3e8d042384859afee5a99c303e014
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[I 2025-09-04 20:49:20,731] Trial 1 finished with value: 0.5888869166374207 and parameters: {'n_estimators': 311, 'max_depth': 5, 'learning_rate': 0.0185964917905873, 'subsample': 0.7871286715029879, 'colsample_bytree': 0.7655448518424737, 'gamma': 0.56485797072793, 'min_child_weight': 6, 'reg_alpha': 2.8574232708356364, 'reg_lambda': 5.907691308199292}. Best is trial 1 with value: 0.5888869166374207.
🏃 View run 1 at: http://127.0.0.1:500

In [54]:
os.makedirs('C:\\Files\\AI\\Projects\\portfolio\\forecasting_orders_timeseries\\mlflow', exist_ok=True)

# === Инициализация клиента MLflow ===
client = MlflowClient(tracking_uri=f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

# Получаем experiment_id по имени
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    raise ValueError(f"Эксперимент '{EXPERIMENT_NAME}' не найден")
experiment_id = experiment.experiment_id

# Ищем лучший run по mean_R2 (сортировка по убыванию, т.к. R² — чем больше, тем лучше)
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.mean_R2 DESC"],  
    max_results=1
)

if not runs:
    print("❌ Не найдено ни одного прогона в эксперименте")
else:
    best_run = runs[0]
    
    # Форматированный вывод
    print("\n" + "="*50)
    print("🔥 ЛУЧШИЙ ПРОГОН:")
    print(f"ID: {best_run.info.run_id}")
    print(f"Название: {best_run.info.run_name}")
    print(f"Метрика mean_R2: {best_run.data.metrics.get('mean_R2', 'недоступно')}")
    print(f"Метрика rmse: {best_run.data.metrics.get('best_mean_RMSE', 'недоступно')}")
    print("\nПАРАМЕТРЫ:")
    
    for param, value in best_run.data.params.items():
        print(f"{param}: {value}")
    
    print("="*50 + "\n")

    # Сохраняем параметры в JSON
    best_params = best_run.data.params
    with open('C:\\Files\\AI\\Projects\\portfolio\\forecasting_orders_timeseries\\mlflow\\best_params.json', 'w', encoding='utf-8') as f:
        json.dump(best_params, f, indent=4, ensure_ascii=False)

    # Приведение типов
    param_types = {
        'n_estimators': int,
        'max_depth': int,
        'learning_rate': float,
        'subsample': float,
        'colsample_bytree': float,
        'gamma': float,
        'min_child_weight': int,
        'reg_alpha': float,
        'reg_lambda': float
    }

    converted_params = {}
    for param, value in best_params.items():
        if param in param_types:
            converted_params[param] = param_types[param](value)
        else:
            converted_params[param] = value


🔥 ЛУЧШИЙ ПРОГОН:
ID: 44ed44c351c84d5f879da28ef349f1e6
Название: 85
Метрика mean_R2: 0.602567195892334
Метрика rmse: недоступно

ПАРАМЕТРЫ:
n_estimators: 385
max_depth: 7
learning_rate: 0.011607262867495543
subsample: 0.6246154044338533
colsample_bytree: 0.6434696326779863
gamma: 0.6195374744428733
min_child_weight: 10
reg_alpha: 0.785574936637615
reg_lambda: 0.950407415765117



In [55]:
# Обучение финальной версии модели
# Создаём модель
final_model = XGBRegressor(**converted_params)

# Обучаем на ВСЕЙ тренировочной выборке
final_model.fit(
    X_train, y_train,
    verbose=False
)

# Предсказание
y_pred_test = final_model.predict(X_test)

# Оценка
test_rmse = mean_squared_error(y_test, y_pred_test)**0.5
test_r2 = r2_score(y_test, y_pred_test)

print(f"✅ Финальная модель обучена и оценена на тесте")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R²: {test_r2:.4f}")

✅ Финальная модель обучена и оценена на тесте
Test RMSE: 39.2881
Test R²: 0.5550


In [56]:
RUN_NAME = 'optuna_final_model'

# Получаем или создаём эксперимент
try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
        print(f"Эксперимент '{EXPERIMENT_NAME}' создан")
    else:
        experiment_id = experiment.experiment_id
except Exception as e:
    print(f"Ошибка с экспериментом: {e}")
    raise

# Запускаем run
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    # Логируем параметры модели вручную
    mlflow.log_params(converted_params)

    # Логируем метрики (только test)
    mlflow.log_metrics({
        "test_rmse": round(test_rmse, 6),
        "test_r2": round(test_r2, 6)
    })
    
    # Логируем артефакты
    # mlflow.log_artifacts(BEST_ASSETS)


    pip_requirements = "C:\\Files\\AI\\Projects\\portfolio\\forecasting_orders_timeseries\\requirements.txt"
    # Для всех категориальных колонок
    # Для числовых
    int_columns = X_test.select_dtypes(include='integer').columns
    X_test[int_columns] = X_test[int_columns].astype('float64')

    # 🔹 Сигнатура и пример
    signature = mlflow.models.infer_signature(X_test, y_pred_test)

    # 🔹 Регистрируем модель
    mlflow.xgboost.log_model(
        xgb_model=final_model,
        artifact_path="model",
        signature=signature,
        input_example=X_test.iloc[:2],
        registered_model_name=REGISTRY_MODEL_NAME,
        # pip_requirements=pip_requirements  
    )

    # Ссылка
    tracking_url = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
    run_url = f"{tracking_url}/#/experiments/{experiment_id}/runs/{run_id}"
    print("\n" + "="*60)
    print("МОДЕЛЬ ЗАЛОГИРОВАНА")
    print(f"Эксперимент: {EXPERIMENT_NAME}")
    print(f"Run ID: {run_id}")
    print(f"Модель: {REGISTRY_MODEL_NAME}")
    print(f" RMSE: {test_rmse:.4f} | R²: {test_r2:.4f}")
    print(f" MLflow UI: {run_url}")

2025/09/04 20:49:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\Andrey\anaconda3\envs\forecasting\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [20:49:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)


Registered model 'forecasting_orders_timeseries_v1' already exists. Creating a new version of this model...
2025/09/04 20:49:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: forecasting_orders_timeseries_v1, version 5



МОДЕЛЬ ЗАЛОГИРОВАНА
Эксперимент: optuna_forecasting
Run ID: 442e3dd068a4455facc96b1b74867ede
Модель: forecasting_orders_timeseries_v1
 RMSE: 39.2881 | R²: 0.5550
 MLflow UI: http://127.0.0.1:5000/#/experiments/1/runs/442e3dd068a4455facc96b1b74867ede
🏃 View run optuna_final_model at: http://127.0.0.1:5000/#/experiments/1/runs/442e3dd068a4455facc96b1b74867ede
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '5' of model 'forecasting_orders_timeseries_v1'.


In [ ]:
# Load model in different ways

# Load as native XGBoost model (preserves all XGBoost functionality)
model = mlflow.xgboost.load_model(f"runs:/{run_id}/model")  # XGBModel
predictions = model.predict(X_test)  
predictions[:5]

array([ 37.137413,  90.062904, 119.89523 , 121.01131 ,  89.995964],
      dtype=float32)